# Laws → Qdrant — Notebook (using `configs.py`)

In [1]:
# Cell 1: Setup and imports
import sys
import os
from pathlib import Path

# Add parent directory to path
notebook_path = Path().resolve()
parent_path = notebook_path.parent
sys.path.append(str(parent_path))

from dotenv import load_dotenv
load_dotenv()

# Verify environment variables
CONGRESS_API_KEY = os.getenv("CONGRESS_API_KEY")
if not CONGRESS_API_KEY:
    raise ValueError("CONGRESS_API_KEY not found in .env")

print("✓ Environment loaded")

✓ Environment loaded


In [2]:
from src.backend.law_parser.congress_laws_parser import CongressAPIClient, CongressLawsQdrantProcessor, BillInfo

print("✓ Imports successful")

✓ Imports successful


In [3]:
# Cell 3: Initialize processor
processor = CongressLawsQdrantProcessor(CONGRESS_API_KEY)

print("✓ Processor initialized")
print(f"✓ Qdrant URL: {processor.qdrant_manager.configs.vectordb.URL}")

2025-10-07 14:28:53,617 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2025-10-07 14:28:53,672 - INFO - HTTP Request: GET http://localhost:6333/collections "HTTP/1.1 200 OK"


✓ Processor initialized
✓ Qdrant URL: http://localhost:6333


In [4]:
# Cell 4: Fetch and filter 10 relevant laws
client = CongressAPIClient(CONGRESS_API_KEY)

all_bills = client.get_enacted_bills([118, 127], limit=200)
print(f"Found {len(all_bills)} enacted bills")

relevant_bills = []
for bill in all_bills[:50]:
    congress = bill.get('congress')
    bill_type = bill.get('type', '').lower()
    bill_number = bill.get('number')
    title = bill.get('title', '')
    
    if not all([congress, bill_type, bill_number]):
        continue
    
    subjects = client.get_bill_subjects(congress, bill_type, bill_number)
    
    if processor.is_relevant_law(subjects, title):
        relevant_bills.append(bill)
        print(f"✓ {len(relevant_bills)}. {title[:80]}...")
        
        if len(relevant_bills) >= 200:
            break

print(f"\nFound {len(relevant_bills)} relevant laws")
print(relevant_bills)

2025-10-07 14:28:55,045 - INFO - Fetching bills from 118th Congress...
2025-10-07 14:29:00,920 - INFO - Found 92 enacted hr bills in 118th Congress
2025-10-07 14:29:05,072 - INFO - Found 32 enacted s bills in 118th Congress
2025-10-07 14:29:09,522 - INFO - Found 5 enacted hjres bills in 118th Congress
2025-10-07 14:29:12,081 - INFO - Fetching bills from 127th Congress...
2025-10-07 14:29:14,714 - INFO - Total enacted bills found: 129


Found 129 enacted bills
✓ 1. Social Security Fairness Act of 2023...
✓ 2. To name the Department of Veterans Affairs community-based outpatient clinic in ...
✓ 3. Supporting America’s Children and Families Act...
✓ 4. To rename the community-based outpatient clinic of the Department of Veterans Af...


2025-10-07 14:30:01,462 - ERROR - API request failed: HTTPSConnectionPool(host='api.congress.gov', port=443): Read timed out. (read timeout=30)


✓ 5. VETS Safe Travel Act...
✓ 6. Think Differently Database Act...
✓ 7. EXPLORE Act...
✓ 8. Recognizing the Importance of Critical Minerals in Healthcare Act of 2023...
✓ 9. To restore the ability of the people of American Samoa to approve amendments to ...
✓ 10. FISHES Act...
✓ 11. Gabriella Miller Kids First Research Act 2.0...

Found 11 relevant laws
[{'congress': 118, 'latestAction': {'actionDate': '2025-01-05', 'text': 'Became Public Law No: 118-273.'}, 'number': '82', 'originChamber': 'House', 'originChamberCode': 'H', 'title': 'Social Security Fairness Act of 2023', 'type': 'HR', 'updateDate': '2025-05-27', 'updateDateIncludingText': '2025-05-27', 'url': 'https://api.congress.gov/v3/bill/118/hr/82?format=json'}, {'congress': 118, 'latestAction': {'actionDate': '2025-01-04', 'text': 'Became Public Law No: 118-259.'}, 'number': '9124', 'originChamber': 'House', 'originChamberCode': 'H', 'title': 'To name the Department of Veterans Affairs community-based outpatient clinic in Aubu

In [5]:
 print(f"\nProcessing: {title[:60]}...")


Processing: FISHES Act...


In [8]:
print(xml_text)
print(type(xml_text))

HR 5103 ENR: Fishery Improvement to Streamline untimely regulatory Hurdles post Emergency Situation Act U.S. House of Representatives text/xml EN Pursuant to Title 17 Section 105 of the United States Code, this file is not subject to copyright protection and is in the public domain. IB One Hundred Eighteenth Congress of the United States of America At the Second Session Begun and held at the City of Washington on Wednesday, the third day of January, two thousand and twenty-four H. R. 5103 AN ACT To require the Director of the Office of Management and Budget to approve or deny spend plans within a certain amount of time, and for other purposes. 1. Short title This Act may be cited as the Fishery Improvement to Streamline untimely regulatory Hurdles post Emergency Situation Act or the FISHES Act . 2. Spend plans Section 312(a)(6) of the Magnuson-Stevens Fishery Conservation and Management Act ( 16 U.S.C. 1861a(a)(6) ) is amended— (1) in subparagraph (D), to read as follows: (D) Spend pla

In [5]:
# Cell 5 (Fixed): Process and store laws in Qdrant
from src.backend.vector_db.chunker import create_chunks

stored_count = 0

for bill in relevant_bills:
    try:
        congress = bill.get('congress')
        bill_type = bill.get('type', '').lower()
        bill_number = bill.get('number')
        title = bill.get('title', '')
        
        print(f"\nProcessing: {title[:60]}...")
        
        xml_url = client.get_bill_xml_url(congress, bill_type, bill_number)
        if not xml_url:
            print("  ✗ No XML found")
            continue
        
        xml_text = client.download_and_clean_xml(xml_url)
        if not xml_text:
            print("  ✗ Failed to download")
            continue
        
        # Use the function directly instead of method
        chunks = create_chunks(xml_text)
        if not chunks:
            print("  ✗ No chunks created")
            continue
            
        latest_action = bill.get('latestAction', {})
        law_number = processor._extract_law_number(latest_action.get('text', ''))
        
        bill_info = BillInfo(
            congress=congress,
            bill_type=bill_type,
            bill_number=bill_number,
            title=title,
            enacted_date=latest_action.get('actionDate'),
            law_number=law_number
        )
        
        law_id = processor._create_law_id(bill_info)
        
        chunks_count = processor.qdrant_manager.process_and_store_law(xml_text, law_id)
        print(f"  ✓ Stored {chunks_count} chunks with ID: {law_id}")
        stored_count += 1
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*60}")
print(f"COMPLETED! Stored {stored_count} laws in Qdrant")
print(f"{'='*60}")


Processing: Social Security Fairness Act of 2023...


2025-10-07 14:30:22,386 - INFO - Found XML URL for HR.82: Enrolled Bill
2025-10-07 14:30:22,388 - INFO - Downloading XML from: https://www.congress.gov/118/bills/hr82/BILLS-118hr82enr.xml
2025-10-07 14:30:23,138 - INFO - Successfully cleaned XML, text length: 2489
2025-10-07 14:30:24,145 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-07 14:30:24,529 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-07 14:30:24,555 - INFO - HTTP Request: PUT http://localhost:6333/collections/laws/points?wait=true "HTTP/1.1 200 OK"


  ✓ Stored 2 chunks with ID: congress_118_hr_82_pl_118_273

Processing: To name the Department of Veterans Affairs community-based o...


2025-10-07 14:30:25,337 - INFO - Found XML URL for HR.9124: Enrolled Bill
2025-10-07 14:30:25,338 - INFO - Downloading XML from: https://www.congress.gov/118/bills/hr9124/BILLS-118hr9124enr.xml
2025-10-07 14:30:26,014 - INFO - Successfully cleaned XML, text length: 3473
2025-10-07 14:30:26,358 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-07 14:30:26,753 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-07 14:30:26,809 - INFO - HTTP Request: PUT http://localhost:6333/collections/laws/points?wait=true "HTTP/1.1 200 OK"


  ✓ Stored 2 chunks with ID: congress_118_hr_9124_pl_118_259

Processing: Supporting America’s Children and Families Act...


...

  ✓ Stored 1 chunks with ID: congress_118_hr_6062_pl_118_232

Processing: FISHES Act...


2025-10-07 14:31:40,461 - INFO - Found XML URL for HR.5103: Enrolled Bill
2025-10-07 14:31:40,462 - INFO - Downloading XML from: https://www.congress.gov/118/bills/hr5103/BILLS-118hr5103enr.xml
2025-10-07 14:31:41,147 - INFO - Successfully cleaned XML, text length: 2648
2025-10-07 14:31:41,786 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-07 14:31:42,642 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-07 14:31:42,659 - INFO - HTTP Request: PUT http://localhost:6333/collections/laws/points?wait=true "HTTP/1.1 200 OK"


  ✓ Stored 2 chunks with ID: congress_118_hr_5103_pl_118_229

Processing: Gabriella Miller Kids First Research Act 2.0...


2025-10-07 14:31:43,347 - INFO - Found XML URL for HR.3391: Enrolled Bill
2025-10-07 14:31:43,348 - INFO - Downloading XML from: https://www.congress.gov/118/bills/hr3391/BILLS-118hr3391enr.xml
2025-10-07 14:31:44,206 - INFO - Successfully cleaned XML, text length: 3046
2025-10-07 14:31:44,666 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-07 14:31:45,120 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-07 14:31:45,139 - INFO - HTTP Request: PUT http://localhost:6333/collections/laws/points?wait=true "HTTP/1.1 200 OK"


  ✓ Stored 2 chunks with ID: congress_118_hr_3391_pl_118_228

COMPLETED! Stored 11 laws in Qdrant


In [6]:
collection_info = processor.qdrant_manager.client.get_collection("laws")
print(f"Collection: {collection_info.status}")
print(f"Total points: {collection_info.points_count}")

2025-10-07 14:31:45,160 - INFO - HTTP Request: GET http://localhost:6333/collections/laws "HTTP/1.1 200 OK"


Collection: green
Total points: 1074
